In [0]:
# %run ../framework

In [0]:
import sys
import os
from pyspark.sql.functions import col

# 1. Add import to silver_unit_test.py
from delta import configure_spark_with_delta_pip

# Import logic that works in GitHub Runner, Local, and Databricks
try:
    from unified_fw.fw import BronzeLayer
except ImportError:
    # 1. check whether __file__  (GitHub Runner/Local Yes, Databricks use os.getcwd())
    if "__file__" in globals():
        current_dir = os.path.dirname(os.path.abspath(__file__))
    else:
        current_dir = os.getcwd()

    # 2. find location project_root by try go back 1 step
    project_root = os.path.abspath(os.path.join(current_dir, ".."))
    package_path = os.path.join(project_root, "logic_packages", "src")

    # If go back but don't find (in case Root directly) use current folder
    if not os.path.exists(package_path):
        package_path = os.path.join(current_dir, "logic_packages", "src")

    # 3. ADD sys.path
    if package_path not in sys.path:
        sys.path.insert(0, package_path)

    from unified_fw.fw import BronzeLayer

# Inject notebook's spark into fw module so classmethods/methods
# that reference bare `spark` (e.g. from_config_table, process_cdf_stream_to_silver)
# can resolve it in the module's global namespace.
import unified_fw.fw as _fw_module
_fw_module.spark = spark

In [0]:
dbutils.widgets.text("pipeline_name", "")
pipeline_name = dbutils.widgets.get("pipeline_name")

In [0]:
if pipeline_name == "" or pipeline_name not in [row.pipeline_name for row in spark.table("workspace.netflix.config_table").select(col("pipeline_name")).collect()]:
    raise Exception("Pipeline name is not in the config table or is empty")

conf = (
    spark.table("workspace.netflix.config_table")
    .filter(col("pipeline_name") == pipeline_name)
    .first()
)

In [0]:
b = BronzeLayer(
    file_path = conf.file_path
    , header = conf.header
    , delimiter = conf.delimiter
    , table_name = conf.table_name
    , schema_detail = conf.schema_detail
)

bronze_df = b.read_from_file()
b.load_to_bronze_table(bronze_df)

In [0]:
# spark.table("workspace.netflix.config_table").display()